# Run published-score comparison (CPU, cluster)

Builds and evaluates the published within-cancer-type prognostic scores (MDCalc-style: mGPS,
RMH, LIPI, ALBI, MELD, CAPRA-mod, and ECOG-free IPI/IMDC/MSKCC) against the held-out full-cohort
text risk score for overall survival. This is a standalone pipeline, independent of
`1_data/01_preprocessing.ipynb` and `4_figures/02_figure_data.ipynb`: it only needs the cohort,
covariate, and full-cohort text risk score outputs those notebooks already produce.

Stages, in order:
1. **Audit** (`pipelines.preprocessing.audit_published_score_inputs`) — read-only. Writes a
   feasibility table; review it before trusting later stages, since a score or stratum can come
   back infeasible for reasons specific to this cohort (lab coverage, missing fields, etc.).
2. **Builder** (`pipelines.preprocessing.build_published_scores`) — one run for the primary
   treatment-anchor/30-day-window case, plus the sequencing-anchor and 90-day-window sensitivity
   runs. Writes `published_scores_df*.csv.gz` and a coverage CSV to `FEATURE_PATH`.
3. **Prep** (`figures.prep.published_scores`) — evaluates published-alone, text-alone, and
   published+text models on shared cohorts/comparable pairs. Writes the five `pubscore_*.csv`
   figure-data outputs consumed by `R/plot_figure_published_scores.R`.

Rendering (`notebooks/4_figures/03_render_figures.R`, already wired to include
`plot_figure_published_scores.R`) is a separate, later step and is not run here.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
print(f"repo root: {REPO_ROOT}")
print(f"Python:  {sys.executable}")

## Run stages

Each stage is invoked as a subprocess so a failure on one stage does not prevent inspecting
what already ran. The audit and primary builder run first so their feasibility/coverage output
is available before the sensitivity builder runs and before prep evaluates any of them.

In [ ]:
STAGES = [
    ["pipelines.preprocessing.audit_published_score_inputs", "--anchor", "treatment"],
    ["pipelines.preprocessing.build_published_scores", "--anchor", "treatment"],
    ["pipelines.preprocessing.build_published_scores", "--anchor", "sequencing"],
    ["pipelines.preprocessing.build_published_scores", "--anchor", "treatment", "--lab-window-days", "90"],
    ["figures.prep.published_scores"],
]


def run_stage(args: list[str]) -> None:
    print("\n=== " + " ".join(args) + " ===", flush=True)
    subprocess.run([sys.executable, "-m", *args], cwd=REPO_ROOT, check=True)


for args in STAGES:
    run_stage(args)

print(f"\nDone. {len(STAGES)} published-score stages succeeded.")